
# Gaussian Process Regression: Cantilever beam model using HODLR


In this example, we create a Gaussian Process Regression metamodel (refer to `gaussian_process_regression`)
of the `cantilever beam <use-case-cantilever-beam>`.
We use a squared exponential covariance kernel for the Gaussian process. In order to estimate the hyper-parameters, we use a design of experiments of size is 20.
Unlike the HMAT variant, the HODLR method does not require an external library.



## Definition of the model



In [ ]:
from openturns.usecases import cantilever_beam
import openturns as ot
import openturns.viewer as otv

We load the cantilever beam use case :



In [ ]:
cb = cantilever_beam.CantileverBeam()

We define the function which evaluates the output depending on the inputs.



In [ ]:
model = cb.model

Then we define the distribution of the input random vector.



In [ ]:
dim = cb.dim  # number of inputs
myDistribution = cb.distribution

## Create the design of experiments



We consider a simple Monte-Carlo sample as a design of experiments. This is why we generate an input sample using the `getSample` method of the distribution.
Then we evaluate the output using the `model` function.



In [ ]:
sampleSize_train = 20
X_train = myDistribution.getSample(sampleSize_train)
Y_train = model(X_train)

The following figure presents the distribution of the vertical deviations `Y` on the training sample.
We observe that the large deviations occur less often.



In [ ]:
histo = ot.HistogramFactory().build(Y_train).drawPDF()
histo.setXTitle("Vertical deviation (cm)")
histo.setTitle("Distribution of the vertical deviation")
histo.setLegends([""])
view = otv.View(histo)

## Create the metamodel



We rely on the `HODLR` (Hierarchical Off-Diagonal Low-Rank) approximation for accelerating the evaluation.
The HODLR algorithm uses a space-filling curve to reorder the rows and columns of the
covariance matrix, then recursively compresses off-diagonal blocks via adaptive cross approximation.
We tune the assembly and recompression tolerances to control the speed/accuracy trade-off.



In [ ]:
ot.ResourceMap.SetAsString("GaussianProcessFitter-LinearAlgebra", "HODLR")
ot.ResourceMap.SetAsScalar("HODLRMatrix-AssemblyEpsilon", 1e-5)
ot.ResourceMap.SetAsScalar("HODLRMatrix-RecompressionEpsilon", 1e-4)

In order to create the GP metamodel, we first select a constant trend with the :class:`~openturns.ConstantBasisFactory` class.
Then we use a squared exponential covariance kernel.
The :class:`~openturns.SquaredExponential` kernel has one amplitude coefficient and 4 scale coefficients.
This is because this covariance kernel is anisotropic : each of the 4 input variables is associated with its own scale coefficient.



In [ ]:
basis = ot.ConstantBasisFactory(dim).build()
covarianceModel = ot.SquaredExponential(dim)

Typically, the optimization algorithm is quite good at setting sensible optimization bounds.
In this case, however, the range of the input domain is extreme.



In [ ]:
print("Lower and upper bounds of X_train:")
print(X_train.getMin(), X_train.getMax())

Finally, we use the :class:`~openturns.GaussianProcessRegression` class to create the GP metamodel.
It requires a training sample, a covariance kernel and a trend basis as input arguments.
We need to set the initial scale parameter for the optimization. The upper bound of the input domain is a sensitive choice here.
We must not forget to actually set the optimization bounds defined above.



In [ ]:
covarianceModel.setScale(X_train.getMax())
fitter = ot.GaussianProcessFitter(X_train, Y_train, covarianceModel, basis)
fitter.run()
fitter_result = fitter.getResult()
algo = ot.GaussianProcessRegression(fitter_result)

The `run` method has optimized the hyperparameters of the metamodel.

We can then print the constant trend of the metamodel, which have been estimated using the least squares method.



In [ ]:
algo.run()
result = algo.getResult()
gpMetamodel = result.getMetaModel()

The `getTrendCoefficients` method returns the coefficients of the trend.



In [ ]:
print(result.getTrendCoefficients())

We can also print the hyperparameters of the covariance model, which have been estimated by maximizing the likelihood.



In [ ]:
result.getCovarianceModel()

## Validate the metamodel



We finally want to validate the Kriging metamodel. This is why we generate a validation sample with size 100 and we evaluate the output of the model on this sample.



In [ ]:
sampleSize_test = 100
X_test = myDistribution.getSample(sampleSize_test)
Y_test = model(X_test)

The :class:`~openturns.MetaModelValidation` class is designed to validate the surrogate models.
To create it, we use a validation sample and a metamodel.



In [ ]:
metamodelPredictions = gpMetamodel(X_test)
val = ot.MetaModelValidation(Y_test, metamodelPredictions)

The :meth:`~openturns.MetaModelValidation.computeR2Score` computes the R2 coefficient of determination.



In [ ]:
r2Score = val.computeR2Score()[0]
print("R2=", r2Score)

The residuals are the difference between the model and the metamodel.



In [ ]:
residualsSample = val.getResidualSample()
graph = ot.HistogramFactory().build(residualsSample).drawPDF()
graph.setXTitle("Residuals (cm)")
graph.setTitle("Distribution of the residuals")
graph.setLegends([""])
view = otv.View(graph)

We observe that the negative residuals occur with nearly the same frequency of the positive residuals: this is a first sign of good quality.



The `drawValidation` method allows one to compare the observed outputs and the metamodel outputs.



sphinx_gallery_thumbnail_number = 3



In [ ]:
graph = val.drawValidation()
graph.setTitle("R2 = %.2f%%" % (100 * r2Score))
view = otv.View(graph)

In [ ]:
otv.View.ShowAll()